# LLM4Rec Aggregated XAI over 20 Random Test Examples

This notebook samples 20 test users once and reuses that exact sample across all
available model variants.

It is the aggregated companion to the single-example XAI notebook. The run is
focused on explanation artifacts only, not ranking metrics.

What it saves:

- Per-example XAI JSON caches
- Per-example summary tables
- Aggregated tables averaged across the 20 sampled users
- Attention rollout grids per model
- Segment heatmaps and layer-importance plots

The LoRA branches are wired in and will run as soon as the local
`llama31-1b-movielens-ranking-lora/` folder and the ranking checkpoints are
present.


## 1. Mount Google Drive


In [ ]:
from google.colab import drive

drive.mount('/content/drive')


## 2. Point the notebook at this repo


In [ ]:
import os
import sys
from pathlib import Path

PROJECT_DIR = '/content/drive/MyDrive/ECS172/project-ADI'  # change if needed
os.chdir(PROJECT_DIR)
ROOT = Path(PROJECT_DIR)
sys.path.insert(0, str(ROOT))

print('Current directory:', ROOT)
print('Top-level files:', [p.name for p in sorted(ROOT.iterdir(), key=lambda p: p.name)[:15]])


## 3. Install dependencies


In [ ]:
!pip install -q -r requirements.txt
!pip install -q captum matplotlib seaborn
!pip install -q --upgrade torchao
print('Dependencies installed.')


## 4. Optional Hugging Face login


In [ ]:
# from huggingface_hub import login
# login('YOUR_HF_TOKEN_HERE')


## 5. Configure the aggregated XAI run


In [ ]:
import json
import math
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from IPython.display import Markdown, display

from llm4rec_xai_pipeline import RankingXAIPipeline, available_configs
from src.data import IdMaps
from src.ranking_data import load_ranking_examples

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 220)

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_theme(style='whitegrid', context='talk')

torch.set_float32_matmul_precision('high')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

CHECKPOINT_DIR = ROOT / 'checkpoints'
OUTPUT_DIR = ROOT / 'xai_outputs'
RUN_ROOT = OUTPUT_DIR / 'colab_20_sample_aggregated_xai'
RUN_ROOT.mkdir(parents=True, exist_ok=True)
RECORD_DIR = RUN_ROOT / 'records'
FIGURE_DIR = RUN_ROOT / 'figures'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

BASE_MODEL = 'unsloth/Llama-3.2-1B-Instruct'
LORA_MODEL = ROOT / 'llama31-1b-movielens-ranking-lora'

SEED = 172
SAMPLE_COUNT = 20
TARGET_MODES = ('ground_truth', 'model_choice')
RUN_MODEL_KEYS = ('base', 'base_sas', 'lora', 'lora_sas')
INCLUDE_ATTENTION_MAPS = True
IG_STEPS = 8
LAYER_STEPS = 4
LAYER_ATTRIBUTION_METHOD = 'feature_ablation'
USE_CHAT_TEMPLATE = False

SEGMENT_ORDER = ('instruction', 'history', 'candidates', 'question', 'continuation', 'other')
SEGMENT_COLUMNS = tuple(f'{{prefix}}:{{segment}}' for prefix in ('IG', 'ROLL', 'ALTI', 'CAM') for segment in SEGMENT_ORDER)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cuda.matmul.allow_tf32 = True

print('Torch dtype:', torch.bfloat16 if DEVICE.type == 'cuda' and getattr(torch.cuda, 'is_bf16_supported', lambda: False)() else torch.float16 if DEVICE.type == 'cuda' else torch.float32)


## 6. Load the test set and sample 20 users once


In [ ]:
id_maps = IdMaps.from_json(CHECKPOINT_DIR / 'id_maps.json')
examples = load_ranking_examples(
    ROOT / 'test_ranking_prompts.json',
    ROOT / 'test_ranking.csv',
    id_maps,
    n_history=10,
)

test_examples = list(examples)
if len(test_examples) < SAMPLE_COUNT:
    raise RuntimeError(f'Need at least {SAMPLE_COUNT} test examples, but only found {len(test_examples)}.')

rng = random.Random(SEED)
sampled_examples = rng.sample(test_examples, SAMPLE_COUNT)
sampled_examples = sorted(sampled_examples, key=lambda ex: int(ex.user_id))

sampled_ids = [int(ex.user_id) for ex in sampled_examples]
(RUN_ROOT / 'sampled_example_ids.json').write_text(json.dumps({'seed': SEED, 'sampled_example_ids': sampled_ids}, indent=2), encoding='utf-8')

print(f'Loaded {len(test_examples)} test examples.')
print(f'Sampled {len(sampled_examples)} users.')
display(pd.DataFrame({'user_id': sampled_ids}))


## 7. Select the model variants that are actually available


In [ ]:
configs = available_configs(
    ROOT,
    CHECKPOINT_DIR,
    base_model=BASE_MODEL,
    lora_model=LORA_MODEL,
)


def config_ready(cfg):
    model_path = Path(str(cfg['model_name']))
    if cfg.get('optional') and not model_path.exists():
        print(f"[skip] {cfg['label']} because {model_path} is missing")
        return False
    if cfg['mode'] == 'candidates':
        emb_path = Path(cfg['embedding_adapter_path'])
        proj_path = Path(cfg['projected_embeddings_path'])
        if not emb_path.exists() or not proj_path.exists():
            missing = emb_path if not emb_path.exists() else proj_path
            print(f"[skip] {cfg['label']} because {missing} is missing")
            return False
    return True

active_configs = [cfg for cfg in configs if cfg['key'] in RUN_MODEL_KEYS and config_ready(cfg)]
print('Active configs:')
for cfg in active_configs:
    print(f"  - {cfg['key']}: {cfg['label']} ({cfg['mode']}) -> {cfg['model_name']}")

if not active_configs:
    raise RuntimeError('No XAI configs are available. Check the model path and checkpoints.')


## 8. Helpers for caching, aggregation, and plots


In [ ]:
def ensure_dir(path):
    path = Path(path)
    path.mkdir(parents=True, exist_ok=True)
    return path


def save_json(data, path):
    path = Path(path)
    ensure_dir(path.parent)
    path.write_text(json.dumps(data, indent=2), encoding='utf-8')


def load_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))

def score_margin(value):
    if isinstance(value, dict):
        return float(value.get('score_margin', value.get('prob', 0.0)))
    return float(value)


def record_path(cfg, example, target_mode):
    return RECORD_DIR / cfg['key'] / target_mode / f'user_{example.user_id}.json'


def add_segment_columns(row, prefix, segment_map):
    for segment in SEGMENT_ORDER:
        row[f'{prefix}:{segment}'] = float(segment_map.get(segment, 0.0))


def flatten_result(result, cfg, target_mode, example):
    row = {
        'config_key': cfg['key'],
        'config_label': cfg['label'],
        'model_name': str(cfg['model_name']),
        'ranking_mode': cfg['mode'],
        'target_mode': target_mode,
        'user_id': int(example.user_id),
        'true_position': int(example.true_position),
        'target_label': result['selection']['target_label'],
        'top_candidate_label': result['selection']['top_candidate_label'],
        'target_candidate_rank': int(result['selection']['target_candidate_rank']),
        'positive_candidate_rank': int(result['selection']['positive_candidate_rank']),
        'top_candidate_rank': int(result['selection']['top_candidate_rank']),
        'top_score': score_margin(result['selection']['top_score']),
        'target_score': score_margin(result['selection']['target_score']),
        'prompt_token_count': int(result['prompt_token_count']),
    }
    add_segment_columns(row, 'IG', result['segment_summary'].get('integrated_gradients', {}))
    if result['segment_summary'].get('attention_rollout') is not None:
        add_segment_columns(row, 'ROLL', result['segment_summary'].get('attention_rollout', {}))
    add_segment_columns(row, 'ALTI', result['segment_summary'].get('alti_plus', {}))
    add_segment_columns(row, 'CAM', result['segment_summary'].get('grad_cam', {}))
    return row


def flatten_layer_rows(result, cfg, target_mode, example):
    rows = []
    for layer_item in result['layer_attributions']:
        rows.append({
            'config_key': cfg['key'],
            'config_label': cfg['label'],
            'model_name': str(cfg['model_name']),
            'ranking_mode': cfg['mode'],
            'target_mode': target_mode,
            'user_id': int(example.user_id),
            'layer_index': int(layer_item['layer_index']),
            'layer_score': float(layer_item['layer_score']),
            'attribution_method': str(layer_item.get('attribution_method', '')),
            'attribution_mode': str(layer_item.get('attribution_mode', '')),
        })
    return rows


def run_config(cfg):
    print()
    print('=' * 88)
    print(f"{cfg['label']} | {cfg['mode']} | {cfg['model_name']}")
    print('=' * 88)

    pipeline = RankingXAIPipeline(
        model_name=cfg['model_name'],
        checkpoint_dir=CHECKPOINT_DIR,
        device=DEVICE,
        freeze_llm=True,
        train_adapter=False,
        load_embedding_adapter=bool(cfg.get('load_embedding_adapter', False)),
        embedding_adapter_path=cfg.get('embedding_adapter_path'),
        projected_embeddings_path=cfg.get('projected_embeddings_path'),
    )

    ensure_dir(RECORD_DIR / cfg['key'])
    rows = []
    layer_rows = []

    for index, example in enumerate(sampled_examples, start=1):
        print(f"  [example] {index}/{len(sampled_examples)} user={example.user_id}")
        for target_mode in TARGET_MODES:
            cache_path = record_path(cfg, example, target_mode)
            if cache_path.exists():
                result = load_json(cache_path)
            else:
                print(f"    [run] target_mode={target_mode}")
                try:
                    result = pipeline.analyze_example(
                        example,
                        mode=cfg['mode'],
                        target_mode=target_mode,
                        ig_steps=IG_STEPS,
                        layer_steps=LAYER_STEPS,
                        layer_attribution_method=LAYER_ATTRIBUTION_METHOD,
                        include_attention_maps=INCLUDE_ATTENTION_MAPS,
                        include_metrics=False,
                    )
                except Exception as exc:
                    print(f"    [error] user={example.user_id} target_mode={target_mode}: {exc}")
                    continue
                save_json(result, cache_path)
            rows.append(flatten_result(result, cfg, target_mode, example))
            layer_rows.extend(flatten_layer_rows(result, cfg, target_mode, example))

    return rows, layer_rows


def plot_segment_heatmap(df, target_mode, method_prefix, out_path):
    subset = df[df['target_mode'] == target_mode].copy()
    if subset.empty:
        return
    segment_cols = [f'{method_prefix}:{segment}' for segment in SEGMENT_ORDER if f'{method_prefix}:{segment}' in subset.columns]
    pivot = subset.groupby('config_label')[segment_cols].mean()
    if pivot.empty:
        return

    fig_height = max(3.5, 0.55 * len(pivot.index) + 1.5)
    fig, ax = plt.subplots(figsize=(10.5, fig_height))
    sns.heatmap(pivot, annot=True, fmt='.1f', cmap='mako', ax=ax, cbar_kws={'label': 'share (%)'})
    ax.set_title(f'{method_prefix} segment shares - {target_mode}')
    ax.set_xlabel('Segment')
    ax.set_ylabel('Model')
    fig.tight_layout()
    fig.savefig(out_path, dpi=180)
    plt.close(fig)


def plot_layer_means(df, target_mode, out_path):
    subset = df[df['target_mode'] == target_mode].copy()
    if subset.empty:
        return
    summary = subset.groupby(['config_label', 'layer_index'], as_index=False)['layer_score'].mean()
    fig, ax = plt.subplots(figsize=(11.5, 5.5))
    sns.lineplot(data=summary, x='layer_index', y='layer_score', hue='config_label', marker='o', ax=ax)
    ax.set_title(f'Average layer importance - {target_mode}')
    ax.set_xlabel('Layer index')
    ax.set_ylabel('Mean layer score')
    ax.grid(True, axis='y', alpha=0.25)
    fig.tight_layout()
    fig.savefig(out_path, dpi=180)
    plt.close(fig)


def plot_attention_grid(records, out_path, title):
    if not records:
        return
    rows = math.ceil(len(records) / 5)
    cols = 5
    vmax = max(float(np.max(np.asarray(rec['xai_result']['attention_maps']['target_row'], dtype=float))) for rec in records)
    fig, axes = plt.subplots(rows, cols, figsize=(18, 3.2 * rows), squeeze=False)
    for axis, record in zip(axes.flat, records):
        row = np.asarray(record['xai_result']['attention_maps']['target_row'], dtype=float)[None, :]
        axis.imshow(row, aspect='auto', cmap='magma', vmin=0.0, vmax=vmax if vmax > 0 else None)
        axis.set_yticks([])
        axis.set_xticks([])
        axis.set_title(f"user {record['example_id']} | {record['selection']['target_label']}", fontsize=9)
    for axis in axes.flat[len(records):]:
        axis.axis('off')
    fig.suptitle(title, fontsize=15)
    fig.tight_layout(rect=[0, 0, 1, 0.96])
    fig.savefig(out_path, dpi=180)
    plt.close(fig)


def summarize_record(record):
    row = {
        'config_key': record['config_key'],
        'config_label': record['config_label'],
        'model_name': record['model_name'],
        'ranking_mode': record['ranking_mode'],
        'target_mode': record['target_mode'],
        'user_id': record['user_id'],
        'true_position': record['true_position'],
        'target_label': record['target_label'],
        'top_candidate_label': record['top_candidate_label'],
        'target_candidate_rank': record['target_candidate_rank'],
        'positive_candidate_rank': record['positive_candidate_rank'],
        'top_candidate_rank': record['top_candidate_rank'],
        'top_score': record['top_score'],
        'target_score': record['target_score'],
        'prompt_token_count': record['prompt_token_count'],
    }
    for prefix in ('IG', 'ROLL', 'ALTI', 'CAM'):
        for segment in SEGMENT_ORDER:
            key = f'{prefix}:{segment}'
            if key in record:
                row[key] = float(record[key])
    return row


## 9. Run the 20-sample aggregated XAI pass


In [ ]:
summary_rows = []
layer_rows = []

for cfg in active_configs:
    try:
        cfg_rows, cfg_layer_rows = run_config(cfg)
        summary_rows.extend(cfg_rows)
        layer_rows.extend(cfg_layer_rows)
    except Exception as exc:
        print(f"[error] config {cfg['label']} failed: {exc}")

summary_df = pd.DataFrame(summary_rows)
layer_df = pd.DataFrame(layer_rows)

summary_df.to_csv(RUN_ROOT / 'summary_table.csv', index=False)
layer_df.to_csv(RUN_ROOT / 'layer_table.csv', index=False)
save_json(summary_df.to_dict(orient='records'), RUN_ROOT / 'summary_table.json')
save_json(layer_df.to_dict(orient='records'), RUN_ROOT / 'layer_table.json')

if summary_df.empty:
    print('[warn] No successful XAI rows were produced, so aggregated tables are empty.')
    aggregated_df = pd.DataFrame()
    aggregated_df.to_csv(RUN_ROOT / 'aggregated_summary.csv', index=False)
    save_json([], RUN_ROOT / 'aggregated_summary.json')
else:
    # Aggregated means across the 20 sampled users.
    exclude = {'config_key', 'config_label', 'model_name', 'ranking_mode', 'target_mode', 'user_id'}
    agg_metric_cols = [c for c in summary_df.columns if c not in exclude]
    aggregated_df = summary_df.groupby(['config_key', 'config_label', 'model_name', 'ranking_mode', 'target_mode'], as_index=False)[agg_metric_cols].mean(numeric_only=True)
    aggregated_df.to_csv(RUN_ROOT / 'aggregated_summary.csv', index=False)
    save_json(aggregated_df.to_dict(orient='records'), RUN_ROOT / 'aggregated_summary.json')

if layer_df.empty:
    layer_agg_df = pd.DataFrame()
    layer_agg_df.to_csv(RUN_ROOT / 'layer_means.csv', index=False)
    save_json([], RUN_ROOT / 'layer_means.json')
else:
    layer_agg_df = layer_df.groupby(['config_key', 'config_label', 'model_name', 'ranking_mode', 'target_mode', 'layer_index'], as_index=False)['layer_score'].mean()
    layer_agg_df.to_csv(RUN_ROOT / 'layer_means.csv', index=False)
    save_json(layer_agg_df.to_dict(orient='records'), RUN_ROOT / 'layer_means.json')

print('Per-example rows:', len(summary_df))
print('Layer rows:', len(layer_df))
print('Aggregated rows:', len(aggregated_df))
display(summary_df.head())


## 10. Plot the aggregated views


In [ ]:
# Heatmaps of segment shares across the 20 sampled users.
for target_mode in TARGET_MODES:
    for method_prefix in ('IG', 'ROLL', 'ALTI', 'CAM'):
        plot_segment_heatmap(
            summary_df,
            target_mode,
            method_prefix,
            FIGURE_DIR / f'{method_prefix.lower()}_segment_heatmap_{target_mode}.png',
        )

# Layer importance plots.
for target_mode in TARGET_MODES:
    plot_layer_means(
        layer_df,
        target_mode,
        FIGURE_DIR / f'layer_means_{target_mode}.png',
    )

# Attention rollout grids for each model under ground-truth mode.
for cfg in active_configs:
    grid_records = []
    for example in sampled_examples:
        cache_path = record_path(cfg, example, 'ground_truth')
        if cache_path.exists():
            grid_records.append(load_json(cache_path))
    plot_attention_grid(
        grid_records,
        FIGURE_DIR / f'attention_grid_{cfg["key"]}_ground_truth.png',
        f'{cfg["label"]} attention rollout grid (ground truth)',
    )

print('Saved figures under:', FIGURE_DIR)


## 11. Aggregated summary tables


In [ ]:
if aggregated_df.empty:
    print('No aggregated rows were produced.')
else:
    cols = [
        'config_label',
        'ranking_mode',
        'target_mode',
        'target_candidate_rank',
        'positive_candidate_rank',
        'top_candidate_rank',
        'top_score',
        'target_score',
    ]
    for prefix in ('IG', 'ROLL', 'ALTI', 'CAM'):
        for segment in SEGMENT_ORDER:
            key = f'{prefix}:{segment}'
            if key in aggregated_df.columns:
                cols.append(key)
    display(aggregated_df[cols])
    display(layer_agg_df.head(20))


## 12. Output summary


In [ ]:
print('XAI cache:')
print(' -', RUN_ROOT / 'records')
print()
print('Per-example tables:')
print(' -', RUN_ROOT / 'summary_table.csv')
print(' -', RUN_ROOT / 'layer_table.csv')
print()
print('Aggregated tables:')
print(' -', RUN_ROOT / 'aggregated_summary.csv')
print(' -', RUN_ROOT / 'layer_means.csv')
print()
print('Figures:')
print(' -', FIGURE_DIR)


## 13. Optional next steps

- Increase `SAMPLE_COUNT` if you want a bigger random aggregate.
- Remove `model_choice` from `TARGET_MODES` if you only want the positive candidate.
- Set `INCLUDE_ATTENTION_MAPS = False` if you want to speed up the run.
